In [107]:
import pandas as pd
import pandas_ta_classic as ta
import config
from oandapyV20 import API
import oandapyV20.endpoints.instruments as instruments
import oandapyV20.endpoints.orders as orders

from datetime import datetime, timezone
import time

In [108]:
#setup your OANDA connection
client = API(access_token=config.OANDA_API_KEY)


In [109]:
timeframe = "M15"
instrument = "GBP_JPY";

In [110]:
def get_candles(tf):
    params = {
        "granularity": tf,
        "price": "A" # ask prices
    }

    r = instruments.InstrumentsCandles(instrument=instrument, params=params)
    candles = client.request(r)['candles']

    # Convert to pandas DataFrame
    data = []

    for c in candles:
        if c["complete"]:
            data.append({
                "time": c["time"],
                "open": float(c["ask"]["o"]),
                "high": float(c["ask"]["h"]),
                "low": float(c["ask"]["l"]),
                "close": float(c["ask"]["c"])
            })

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    
    return df

In [111]:
# function to calculate indicators
def calculate_indicators(df):
    # EMAs
    df["EMA_5"] = ta.ema(df["close"], length=5)
    df["EMA_8"] = ta.ema(df["close"], length=8)

    # ATR Average True Range
    df["ATR_14"] = ta.atr(df["high"], df["low"], df["close"], length=14)

    return df

In [112]:
# function to place an order
def place_order(stop_loss, take_profit):
    data = {
        "order": {
            "instrument": instrument,
            "units": 1,
            "type": "MARKET",
            "stopLossOnFill": {"price": f"{stop_loss:.3f}"},
            "takeProfitOnFill": {"price": f"{take_profit:.3f}"}
        }
    }

    r = orders.OrderCreate(config.OANDA_ACCOUNT_ID, data=data)

    client.request(r)
    print(f"Placed order for {instrument} with stop loss: {round(stop_loss, 3)} and take profit: {round(take_profit, 3)}")

In [113]:
# Function to check for EMA Crossover Signals
def ema_crossover(df):
    tp_ratio = 1.5

    # Check if the 5-peiord EMA crosses above the 8-period EMA (Bullish Crossover)
    last_candle = df.iloc[-1]
    prev_candle = df.iloc[-2]

    # Crossover Buy Signal (EMA 5 crosses above EMA 8)
    if prev_candle["EMA_5"] < prev_candle["EMA_8"] and last_candle["EMA_5"] > last_candle["EMA_8"]:
        print("Bullish Crossover Detected - Potential Buy Signal")
        entry_price = last_candle["close"]
        stop_loss = entry_price - last_candle["ATR_14"]  # ATR-based
        stop_distance = entry_price - stop_loss
        take_profit = entry_price + (stop_distance * tp_ratio)  # Risk-Reward
        place_order(stop_loss, take_profit)
    else:
        print("No Crossover Detected - No Trade Signal")

In [ ]:
def run_bot():
    print("Running Bot...")
    last_checked = None

    while True:
        current_time = datetime.now(timezone.utc)
        
        # Check for a new candle based on my timeframe
        if current_time.minute % 1 == 0 and current_time.second < 10:
            # Check if the interval has changed since the last check
            if last_checked != current_time.minute:
                print("Checking for trade signals...")
                price = get_candles(timeframe)
                price = calculate_indicators(price)
                ema_crossover(price)
                last_checked = current_time.minute # update to prevent re-triggering
        time.sleep(1)

In [115]:
run_bot()

Running Bot...
Checking for trade signals...
No Crossover Detected - No Trade Signal
Checking for trade signals...
No Crossover Detected - No Trade Signal
Checking for trade signals...
No Crossover Detected - No Trade Signal
Checking for trade signals...
No Crossover Detected - No Trade Signal
Checking for trade signals...
No Crossover Detected - No Trade Signal
Checking for trade signals...
No Crossover Detected - No Trade Signal


KeyboardInterrupt: 